In [1]:
import boto3
import json
import time
import pandas as pd
import argparse 
import random

# session = boto3.Session(profile_name='corp-us-east-1')
session = boto3.Session(profile_name='default')


# Define the table name
table_name = 'prompt_hub_table'
# Create DynamoDB resource
dynamodb = session.resource('dynamodb')
table = dynamodb.Table(table_name)

In [2]:

def filter_items(field,val):
    # Scan the table to get all items where is_external is true
    response = table.scan(
        FilterExpression=f'{field} = :val',
        ExpressionAttributeValues={':val': val}
    )
    
    items = response['Items']
    print(f"found:{len(items)}")
    
    # Handle pagination if there are more items
    while 'LastEvaluatedKey' in response:
        response = table.scan(
            FilterExpression='{field} = :val',
            ExpressionAttributeValues={':val': val},
            ExclusiveStartKey=response['LastEvaluatedKey']
        )
        items.extend(response['Items'])
    return items

def update_items(items,field,val):
    # Update each item's delete_status
    updated_count = 0
    for item in items:
        # Get the primary key values from your item
        # Modify these according to your table's primary key structure
        key = {
            'id': item['id']  # Assuming 'id' is your primary key
            # Add other key attributes if you have a composite key
        }

        # Update the item
        table.update_item(
            Key=key,
            UpdateExpression=f'SET {field} = :val',
            ExpressionAttributeValues={
                ':val': val
            }
        )
        updated_count += 1

    print(f"Successfully updated {updated_count} items")
    return updated_count

In [5]:
# 把is_external = true都删除
items = filter_items("is_external",True)
update_items(items, "delete_status","deleted")

found:26
Successfully updated 26 items


26

In [5]:
# 把is_recommended = true 改成 is_recommended = False
# items = filter_items("is_recommended",False)
# update_items(items, "is_recommended",True)

## upload new template

In [3]:
def upload_to_dynamodb(table_name, json_data):
    dynamodb = session.resource('dynamodb')
    table = dynamodb.Table(table_name)

    for item in json_data:
        table.put_item(Item=item)

    print(f"Data uploaded to DynamoDB table: {table_name}")


def generate_id():
    timestamp = int(time.time() * 1000)  # Get the current timestamp in milliseconds
    random_number = str(random.randint(0, 16**6))  # Generate a random 6-digit number
    return f"{timestamp}-{random_number}"


def process_excel(filename):
    df = pd.read_excel(filename)
    time_tuple = time.localtime( time.time())
    createtime = time.strftime("%Y-%m-%d %H:%M:%S", time_tuple)
    df.drop(['Cx Cases in Prod'],inplace=True,axis=1)
    df.rename(columns={'Category':'category',
                       'Name':'demo_name',
                       'Description':'description',
                       'Further Support':'further_support',
                       'Status':'demo_type',
                       'Simple Demo Introduction Deck':'deck_link',
                       'Demo Video Link':'demo_link',
                       'Code Repo Link':'code_repo_link',
                       'China Region Support':'china_region_support',
                       'Contact':'contact',
                       'Team':'team',
                       'Industry':'industry'
                       }, inplace=True)
    df['createtime'] = createtime
    df['company'] = 'default'
    df['template'] = ''
    df['id'] = df.apply(lambda x: generate_id(), axis=1)
    df['demo_version'] = '2025v1'
    # df['industry'] = df.apply(lambda x: [{"label":i,"value":i}  for i in x['industry'].split('|') ],axis=1)
    
    df_dict = json.loads(df.to_json(orient='index'))
    return  list(df_dict.values())


In [4]:


filename = "[0115-Template] GenAI Horizontal Asset Hub.xlsx"

json_data = process_excel(filename)
print(json_data)

[{'category': '翻译', 'demo_name': '基于LLM的专词翻译方案', 'description': '本专词翻译方案致力于提升专有名词的翻译质量。该方案避免了将所有专词都放入提示词的做法，提高了翻译效率。它允许用户自定义翻译规则，尤其适合专词较多的场景。', 'further_support': 'YES', 'demo_type': 'Ready-to-Adopt Asset', 'deck_link': 'https://aws.highspot.com/items/6773e1d0ae50e7759c31c662?lfrm=shp.1', 'demo_link': 'https://aws.highspot.com/items/6764dcfd156457a800fc4a9c?lfrm=shp.0', 'code_repo_link': 'https://github.com/aws-samples/rag-based-translation-with-dynamodb-and-bedrock', 'china_region_support': 'YES', 'contact': 'ybalbert@amazon.com', 'createtime': '2025-01-15 15:43:52', 'company': 'default', 'template': '', 'id': '1736955832526-3070936', 'demo_version': '2025v1'}, {'category': '角色扮演', 'demo_name': '提示词自动调优工具', 'description': '本方案主打生成批量角色提示词和实现客制化自动测评角色输出, 本方案提供两个模块功能：1、提示优化器：根据设定的人类偏好，基于测试对话的评估来迭代角色提示。\xa02、提示分析器：这个模块追踪整个优化过程中对提示所做的更改。这样用户可以更好地理解基于评分对角色提示做出了哪些更改。', 'further_support': 'YES', 'demo_type': 'Simple Demo Asset', 'deck_link': 'https://aws.highspot.com/items/6784c0f072bf52feb1a

In [5]:
# Upload the JSON data to a new DynamoDB table
upload_to_dynamodb(table_name, json_data)
print(f'uploded data from {filename}')

Data uploaded to DynamoDB table: prompt_hub_table
uploded data from [0115-Template] GenAI Horizontal Asset Hub.xlsx
